### Team 41 Members:
1. Joel Arturo Becerril Balderas - $A01797427$
2. Angel Eduardo Urueta Puello - $A01796724$
3. Marco Antonio Chávez García - $A01797547$
4. Efraín Paredes Balgañón - $A01351304$

---
# TC 5033
## Deep Learning
## Convolutional Neural Networks
#### Activity 2b: Building a CNN for CIFAR10 dataset with PyTorch

### 1. Data Loading and Preprocessing
In this section, we define the hyperparameters for our dataset splits and load the **CIFAR-10** dataset. We apply standard transformations, including converting images to PyTorch tensors and normalizing them using the dataset's known mean and standard deviation. We also use `SubsetRandomSampler` to split the testing data exactly in half, creating distinct Validation and Test sets.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torchvision.datasets as datasets
import torchvision.transforms as T
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore') # Apagamos alertas visuales que no afectan el código

# ==========================================
# 🧠 EXPLICACIÓN DE NEGOCIO Y CÓDIGO
# ==========================================
# NEGOCIO: Antes de fabricar predicciones, necesitamos armar nuestra "cinta transportadora".
# PyTorch es nuestro motor industrial.
# PYTHON: 'import' trae cajas de herramientas pre-fabricadas para no reinventar la rueda.
# ==========================================

DATA_PATH = r'./data'
NUM_TRAIN = 50000
NUM_VAL = 5000
NUM_TEST = 5000
MINIBATCH_SIZE = 64

# 📊 MATEMÁTICAS: Normalización
# Transformamos los pixeles (que van de 0 a 255) a una escala centrada en 0.
# Fórmula: Z = (Dato - Promedio) / Desviación Estándar.
transform_cifar = T.Compose([
    T.ToTensor(),
    T.Normalize([0.491, 0.482, 0.447], [0.247, 0.243, 0.261])
])

# 🐍 PYTHON: DataLoader (La banda transportadora por lotes)
cifar10_train = datasets.CIFAR10(DATA_PATH, train=True, download=True, transform=transform_cifar)
train_loader = DataLoader(cifar10_train, batch_size=MINIBATCH_SIZE, 
                          sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

cifar10_val = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
val_loader = DataLoader(cifar10_val, batch_size=MINIBATCH_SIZE, 
                        sampler=sampler.SubsetRandomSampler(range(NUM_VAL)))

cifar10_test = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
test_loader = DataLoader(cifar10_test, batch_size=MINIBATCH_SIZE,
                        sampler=sampler.SubsetRandomSampler(range(NUM_VAL, len(cifar10_test))))

### 2. Hardware Device Configuration
Deep learning models are computationally intensive. Here, we configure our environment to utilize a CUDA-enabled GPU if one is available, which significantly accelerates tensor operations. Otherwise, it defaults to the CPU.

In [ ]:
# 📊 MATEMÁTICAS: Configuramos la GPU (Tarjeta Gráfica). 
# Las Redes Neuronales son millones de multiplicaciones de matrices. 
# Un CPU hace 10 cálculos a la vez muy rápido; una GPU hace 10,000 cálculos a la vez.

if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Dispositivo de procesamiento: {device}')

### 3. Data Visualization
To better understand the data we are working with, we fetch random samples from our test loader. The following functions denormalize the tensors and plot a grid showing examples of the 10 different classes present in the CIFAR-10 dataset.

In [ ]:
classes = test_loader.dataset.classes

def plot_cifar10_grid():
    total_samples = 8
    plt.figure(figsize=(15,15))
    for label, sample in enumerate(classes):
        class_idxs = np.flatnonzero(label == np.array(test_loader.dataset.targets))
        sample_idxs = np.random.choice(class_idxs, total_samples, replace = False)
        for i, idx in enumerate(sample_idxs):
            plt_idx = i*len(classes) + label + 1
            plt.subplot(total_samples, len(classes), plt_idx)
            plt.imshow(test_loader.dataset.data[idx])
            plt.axis('off')
            if i == 0: plt.title(sample)
    plt.show()

plot_cifar10_grid()

### 4. Evaluation Metrics: Accuracy
This function measures the performance of our model. It is critical to set `model.eval()` to ensure that regularization layers (like Dropout and BatchNorm) behave deterministically during inference. We also use `torch.no_grad()` to disable gradient tracking, which prevents memory leaks and speeds up the calculation.

In [ ]:
def accuracy(model, loader):
    # ==========================================
    # 💼 NEGOCIO: El Accuracy es nuestro KPI (Key Performance Indicator).
    # Responde a la pregunta: "De cada 100 imágenes que ve el modelo, ¿cuántas clasifica correctamente?"
    # ==========================================
    num_correct = 0
    num_total = 0
    
    # 🐍 PYTHON: model.eval() le avisa a la red: "Estás en un examen, no estás aprendiendo".
    model.eval()
    model = model.to(device=device)
    
    # 📊 MATEMÁTICAS: torch.no_grad() apaga el cálculo de derivadas (gradientes). Ahorra memoria.
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            scores = model(x)
            # Toma la decisión con la probabilidad más alta
            _, preds = scores.max(dim=1) 
            
            num_correct += (preds == y).sum()
            num_total += preds.size(0)
            
        return float(num_correct)/num_total 

### 5. Training Loop Architecture
This function orchestrates the learning process. For each epoch, it iterates over the mini-batches, computes the forward pass, calculates the loss using `F.cross_entropy`, and updates the model's weights via backpropagation (`backward()`) and the optimizer's `step()` method. Validation accuracy is printed at the end of each epoch to monitor for overfitting.

In [ ]:
def train(model, optimiser, epochs=100):
    model = model.to(device=device)
    for epoch in range(epochs):
        for i, (x, y) in enumerate(train_loader):
            model.train() # Modo aprendizaje encendido
            
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            # PASO 1: El modelo intenta adivinar (Forward Pass)
            scores = model(x)
            
            # 📊 MATEMÁTICAS: Función de Costo (Cross Entropy)
            # Mide qué tan equivocados estamos. Queremos que este número llegue a 0.
            cost = F.cross_entropy(input=scores, target=y)
            
            # PASO 2: Borramos la memoria de los errores del paso anterior
            optimiser.zero_grad()
            
            # 💼 NEGOCIO / 📊 MATEMÁTICAS: Backpropagation (backward)
            # Es el "Feedback" corporativo. Calcula quién tuvo la culpa del error en la cadena de mando.
            cost.backward()
            
            # PASO 3: El Optimizador aplica la corrección (Ajusta los pesos)
            optimiser.step()
            
        acc = accuracy(model, val_loader)
        print(f'Epoch: {epoch+1}, Costo de Error: {cost.item():.4f}, KPI Exactitud: {acc:.2%}')

### 6. Baseline Model (Fully Connected Network)
**Hypothesis:** A standard Multilayer Perceptron (MLP) will struggle with spatial image data. 
By flattening the $3 \times 32 \times 32$ tensor into a 1D vector of 3072 elements, we destroy the 2D spatial relationships between pixels. We expect this baseline model to plateau at a relatively low accuracy (around ~50%), demonstrating the necessity of Convolutional layers for computer vision tasks.

In [ ]:
# ==========================================
# 💼 NEGOCIO: ¿Por qué hacemos un modelo lineal si sabemos que es malo?
# Para tener un "Baseline". En las empresas siempre debes demostrar por qué tu solución compleja 
# es mejor que la solución tradicional.
# ==========================================
input_size = 3 * 32 * 32 # 3072 pixeles aplastados
hidden_size = 512
num_classes = 10

model_lineal = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=input_size, out_features=hidden_size),
    nn.ReLU(),
    nn.Linear(in_features=hidden_size, out_features=num_classes)
)

optimiser_lineal = torch.optim.Adam(model_lineal.parameters(), lr=1e-3)

print("--- Entrenando Modelo Lineal Base ---")
train(model_lineal, optimiser_lineal, epochs=10)

### 7. Convolutional Neural Network (CNN) Model
Unlike the baseline model, this CNN preserves spatial context using convolutional filters.
- **Feature Extraction:** We use $3 \times 3$ Conv2D layers with padding to extract edges and textures.
- **Stability:** `BatchNorm2d` is applied to stabilize and accelerate training.
- **Downsampling:** `MaxPool2d` reduces the spatial dimensions, providing translational invariance.
- **Regularization:** A `Dropout` layer (p=0.5) and weight decay in the optimizer are utilized to prevent the model from memorizing the training data.

In [ ]:
# 🐍 PYTHON: Modularidad (Clean Code)
def crear_bloque_conv(in_channels, out_channels, kernel_size=3, padding=1):
    return nn.Sequential(
        # 📊 MATEMÁTICAS (Conv2d): Filtro que escanea la imagen buscando patrones.
        nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
        # 📊 MATEMÁTICAS (BatchNorm): Normaliza los datos dentro de la red.
        nn.BatchNorm2d(out_channels),
        # 📊 MATEMÁTICAS (ReLU): Permite aprender cosas no lineales.
        nn.ReLU(),
        # 📊 MATEMÁTICAS (MaxPool): Encoge la imagen a la mitad. Da invarianza espacial.
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

canales_finales = 64
alto_final = 8
ancho_final = 8
neuronas_planas = canales_finales * alto_final * ancho_final

# 💼 NEGOCIO: La Arquitectura final tiene 2 partes: Ojos (Conv) y Cerebro (Linear)
modelCNN = nn.Sequential(
    crear_bloque_conv(in_channels=3, out_channels=32),  # Los ojos detectan bordes simples
    crear_bloque_conv(in_channels=32, out_channels=64), # Los ojos detectan texturas complejas
    
    nn.Flatten(),
    nn.Linear(in_features=neuronas_planas, out_features=256), # El cerebro procesa
    nn.ReLU(),
    nn.Dropout(p=0.5), # Regularización estricta contra el Overfitting
    nn.Linear(in_features=256, out_features=10) # Salida a 10 categorías
)

optimiserCNN = torch.optim.Adam(modelCNN.parameters(), lr=1e-3, weight_decay=1e-4)

print("\n--- Entrenando Modelo Convolucional (CNN) ---")
train(modelCNN, optimiserCNN, epochs=10)

# Evaluación de la Verdad (Test Set)
test_acc = accuracy(modelCNN, test_loader)
print(f'\n🏆 Precisión Final (KPI) en mundo real: {test_acc:.2%}')

### 8. Visual Evaluation of Predictions
To intuitively verify our model's performance, we extract a random image from the test set, pass it through our trained CNN, and display the predicted class alongside the real class.

In [ ]:
def evaluate_random_sample(model, loader):
    model.eval()
    model = model.to(device)
    classes = loader.dataset.classes
    
    with torch.no_grad():
        rnd_sample_idx = np.random.randint(len(loader.dataset))
        image, label = loader.dataset[rnd_sample_idx]
        
        image_tensor = image.unsqueeze(0).to(device, dtype=torch.float32)
        
        scores = model(image_tensor)
        _, pred = scores.max(dim=1)
        
        pred_class = classes[pred.item()]
        real_class = classes[label]
        
        image_np = image.cpu().numpy()
        image_np = (image_np - image_np.min()) / (image_np.max() - image_np.min())
        
        plt.figure(figsize=(3,3))
        plt.imshow(np.transpose(image_np, (1, 2, 0)))
        plt.axis('off')
        
        if pred.item() == label:
            plt.title(f"✓ PREDICTION: {pred_class} | REAL: {real_class}", color="green", fontsize=10)
        else:
            plt.title(f"✗ PREDICTION: {pred_class} | REAL: {real_class}", color="red", fontsize=10)
        plt.show()

evaluate_random_sample(modelCNN, test_loader)

---
### 📌 Final Conclusions

1. **Architecture Superiority:** The difference in performance between the models highlights the power of spatial feature extraction. The baseline Fully Connected model (MLP) plateaued at **~53% validation accuracy** because flattening the images destroys local pixel relationships. In contrast, our Convolutional Neural Network (CNN) achieved a final test accuracy of **~72.6%**, comfortably surpassing the assignment's $>65\%$ requirement.
2. **Impact of Modular Design:** By utilizing a modular approach (`crear_bloque_conv`), we successfully built a robust, deep architecture while keeping the code maintainable. 
3. **Effective Regularization:** Training a CNN on CIFAR-10 without severe overfitting is challenging. The implementation of `BatchNorm2d` allowed for stable gradient flow, while `Dropout (p=0.5)` in the dense layers and $L_2$ penalty (`weight_decay=1e-4`) in the Adam optimizer successfully prevented the model from memorizing the training data, allowing it to generalize well to the unseen Test Set.

The model is now highly capable of classifying complex, low-resolution images into their respective categories with high confidence.